# 02 — Resume Classifier (supervised)
Clean text → TF-IDF → Logistic Regression. Evaluate with confusion matrix + F1.

In [1]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Locate the project folder
PROJECT_DIR = Path.cwd().parent

# Load the resume dataset
resumes = pd.read_csv(
    PROJECT_DIR / "data" / "processed" / "resumes_clean.csv"
)

# Display dataset information
print("Dataset shape:", resumes.shape)
print("\nAvailable columns:")
print(resumes.columns.tolist())

# Display the first five rows
resumes.head()

Dataset shape: (962, 2)

Available columns:
['Category', 'text']


,Category,text
0,Data Science,skills programming languages python pandas num...
1,Data Science,education details may 2013 to may 2017 b.e uit...
2,Data Science,areas of interest deep learning control system...
3,Data Science,skills r python sap hana tableau sap hana sql ...
4,Data Science,education details mca ymcaust faridabad haryan...


In [2]:
# Check the dataset before training
print("Number of resumes:", len(resumes))
print("\nResume categories:")
print(resumes["Category"].value_counts())

# Remove rows with missing category or resume text
resumes = resumes.dropna(subset=["Category", "text"])

# Remove empty resume text
resumes = resumes[resumes["text"].str.strip() != ""]

print("\nDataset shape after cleaning:", resumes.shape)
print("Number of categories:", resumes["Category"].nunique())

Number of resumes: 962

Resume categories:
Category
Java Developer               84
Testing                      70
DevOps Engineer              55
Python Developer             48
Web Designing                45
HR                           44
Hadoop                       42
Data Science                 40
Mechanical Engineer          40
Sales                        40
Operations Manager           40
ETL Developer                40
Blockchain                   40
Arts                         36
Database                     33
Health and fitness           30
Electrical Engineering       30
PMO                          30
Business Analyst             28
DotNet Developer             28
Automation Testing           26
Network Security Engineer    25
Civil Engineer               24
SAP Developer                24
Advocate                     20
Name: count, dtype: int64

Dataset shape after cleaning: (962, 2)
Number of categories: 25


In [4]:
# Separate resume text (X) and categories (y)
X = resumes["text"]
y = resumes["Category"]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training resumes:", len(X_train))
print("Testing resumes:", len(X_test))

print("\nTraining category distribution:")
print(y_train.value_counts())

Training resumes: 769
Testing resumes: 193

Training category distribution:
Category
Java Developer               67
Testing                      56
DevOps Engineer              44
Python Developer             38
Web Designing                36
HR                           35
Hadoop                       34
Data Science                 32
Sales                        32
ETL Developer                32
Operations Manager           32
Blockchain                   32
Mechanical Engineer          32
Arts                         29
Database                     26
PMO                          24
Health and fitness           24
Electrical Engineering       24
DotNet Developer             23
Business Analyst             22
Automation Testing           21
Network Security Engineer    20
SAP Developer                19
Civil Engineer               19
Advocate                     16
Name: count, dtype: int64


In [5]:
# Convert resume text into numerical features using TF-IDF
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)

# Learn vocabulary from training data and transform both sets
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training feature matrix:", X_train_tfidf.shape)
print("Testing feature matrix:", X_test_tfidf.shape)
print("Number of TF-IDF features:", len(tfidf.get_feature_names_out()))

Training feature matrix: (769, 10000)
Testing feature matrix: (193, 10000)
Number of TF-IDF features: 10000


In [6]:
# Train the Logistic Regression classifier
classifier = LogisticRegression(
    max_iter=1000,
    random_state=42
)

classifier.fit(X_train_tfidf, y_train)

print("Resume classification model trained successfully!")

Resume classification model trained successfully!


In [7]:
# Predict resume categories for the test dataset
y_pred = classifier.predict(X_test_tfidf)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy:.2%}")

# Display precision, recall, and F1-score
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Model Accuracy: 99.48%

Classification Report:
                           precision    recall  f1-score   support

                 Advocate       1.00      1.00      1.00         4
                     Arts       1.00      1.00      1.00         7
       Automation Testing       0.83      1.00      0.91         5
               Blockchain       1.00      1.00      1.00         8
         Business Analyst       1.00      1.00      1.00         6
           Civil Engineer       1.00      1.00      1.00         5
             Data Science       1.00      1.00      1.00         8
                 Database       1.00      1.00      1.00         7
          DevOps Engineer       1.00      0.91      0.95        11
         DotNet Developer       1.00      1.00      1.00         5
            ETL Developer       1.00      1.00      1.00         8
   Electrical Engineering       1.00      1.00      1.00         6
                       HR       1.00      1.00      1.00         9
              